**A. Import SparkSession & Extract**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum

spark = SparkSession.builder \
    .appName("Tugas6_Dany_Akhdan_Imaduddin") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_prod = spark.read.json("tugas6_produk.json")
df_rev = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print(f"Data Transaksi: {df_trx.count()} baris")
df_trx.printSchema()
print(f"Data Produk: {df_prod.count()} baris")
df_prod.printSchema()
print(f"Data Ulasan: {df_rev.count()} baris")
df_rev.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/25 21:05:46 WARN Utils: Your hostname, danyai, resolves to a loopback address: 127.0.1.1; using 192.168.1.195 instead (on interface wlo1)
26/09/25 21:05:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/dany/anaconda3/envs/bigdata/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/25 21:05:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Data Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. Transform - Join**

In [2]:
# B. Join Data & Hitung Pendapatan
df_join = df_trx.join(df_prod, on="product_id", how="inner") \
                .join(df_rev, on="order_id", how="left")

df_join = df_join.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

**C. Transform - Solving**

In [3]:
# C. Penanganan Data Kosong
df_join = df_join.withColumn("ada_ulasan", col("rating").isNotNull())
df_join = df_join.fillna({"rating": 0})

df_join.show(5)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|ada_ulasan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|      true|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|     0|         1500000|     false|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|      true|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|      true|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|     0|          450000|     false|
+--------+----------+---

**D. Load**

In [4]:
# D. Load ke HDFS dan partisi by kategori
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

df_join.write.mode("overwrite").partitionBy("kategori").parquet(output_path)

!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

df_hasil_hdfs = spark.read.parquet(output_path)
print("Total baris tersimpan di HDFS:", df_hasil_hdfs.count())

-rw-r--r--   3 dany supergroup          0 2026-09-25 21:06 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - dany supergroup          0 2026-09-25 21:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 dany supergroup      16385 2026-09-25 21:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-9cba19f5-0bfa-40f0-948c-bfee3dc3ba01.c000.snappy.parquet
drwxr-xr-x   - dany supergroup          0 2026-09-25 21:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 dany supergroup      11159 2026-09-25 21:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-9cba19f5-0bfa-40f0-948c-bfee3dc3ba01.c000.snappy.parquet
drwxr-xr-x   - dany supergroup          0 2026-09-25 21:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 dany supergroup      10419 2026-09-25 21:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-9cba19f5-0bfa-40f0-948c-bfee3dc3ba01.c000.snappy.parquet
drwxr-xr-x   - dany supergroup 

**E. Insight**

**Display Review**

In [5]:
# Kalkulasi persentase ulasan
df_hasil_hdfs.groupBy("kategori") \
    .agg((spark_sum(col("ada_ulasan").cast("int")) / count("*") * 100).alias("persentase_ulasan")) \
    .orderBy("persentase_ulasan").show()

+------------+-----------------+
|    kategori|persentase_ulasan|
+------------+-----------------+
|     Makanan|68.84328358208955|
|   Kesehatan|68.88657648283039|
|     Fashion|70.14492753623188|
|Rumah Tangga| 70.5521472392638|
|  Elektronik|  70.659407138536|
+------------+-----------------+



**Berdasarkan hasil pipeline ETL, kategori produk Makanan memiliki tingkat partisipasi ulasan terendah. Temuan ini krusial bagi tim pemasaran karena minimnya ulasan dapat menghambat konversi pelanggan baru yang sangat bergantung pada rating sebelum membeli. Manajemen perlu meluncurkan kampanye insentif khusus pada kategori tersebut untuk mendongkrak jumlah ulasan pasca-transaksi.**